In [ ]:
# import the required packages 
import ollama
import pandas as pd
import os
import time
from tqdm import tqdm

In [ ]:
# Path of the  data to be labeled(input_data_path)
input_data_path = "/Users/jisha/Desktop/Sarcasm_Final/Exploratory_Data_Analysis/final_bitcoin_data.csv"
# Path where the results to be saved
output_data_path = "/Users/jisha/Desktop/Sarcasm_Final/Data_Labelling/outputs.csv"

In [ ]:
# read the input data to a pandas dataframe and check the information about its contents 

input_data = pd.read_csv(input_data_path)
input_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 906 entries, 0 to 905
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   label           906 non-null    int64 
 1   comment         906 non-null    object
 2   author          906 non-null    object
 3   subreddit       906 non-null    object
 4   score           906 non-null    int64 
 5   ups             906 non-null    int64 
 6   downs           906 non-null    int64 
 7   date            906 non-null    object
 8   created_utc     906 non-null    object
 9   parent_comment  906 non-null    object
dtypes: int64(4), object(6)
memory usage: 70.9+ KB


In [ ]:
# Check if the path exists, if yes then read the data to new_labeled_df and check if the column llama_temp1 exists, 
# else create a new dataframe by copying the contents from the input data and create a new colukn named llama_temp1

if os.path.exists(output_data_path):
    print("Found an existing file with the same name, continuing labeling...")
    new_labeled_df = pd.read_csv(output_data_path)
    
    if 'llama_temp0' not in new_labeled_df.columns:
        new_labeled_df['llama_temp0'] = pd.NA
else:
    print("File not found with the given name, creating a new file and starting labeling from the beginning")
    new_labeled_df = input_data.copy()
    new_labeled_df['llama_temp0'] = pd.NA

def index_finder(df):
    for index, value in df['llama_temp0'].items():
        if pd.isna(value):
            return index
        if value not in [0, 1, 0.0, 1.0]:
            return index
    return len(df)

start_index = index_finder(new_labeled_df)

if start_index >= len(new_labeled_df):
    print("\n The dataset is already fully labeled. No further labeling is required.")
else:
    print(f"\n Continuing labelling from row index: {start_index} out of {len(new_labeled_df)}")

Found an existing file with the same name, continuing labeling...

 The dataset is already fully labeled. No further labeling is required.


In [ ]:
# Function to label the data

def labeller(parent_comment, comment):
    prompt = f"""
    Find out if the reply comment is a sarcastic response to the parent comment.
    - The Parent comment: "{parent_comment}"
    - The Reply comment: "{comment}"

    The output should only contain 1 if the reply comment is sarcastic and 0 if the reply comment is nonsarcastic. Do not include any explanation. It should strictly be a 0 or 1.
    """
    try:
        # Call the llama3.2:3b model with temprature 0.0
        result = ollama.chat(
            model='llama3.2:3b', 
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.0} 
        )
        original_response = result['message']['content'].strip()
        # Select only the required information from the result given by the model which is 0 or 1 

        if "1" in original_response:
            return 1
        elif "0" in original_response:
            return 0
        else:
            return 1 if "sarcastic" in original_response.lower() else 0
            
    except Exception as e:
        raise RuntimeError(f"Ollama Call Error: {e}")

In [ ]:
# Strat labeling the data from the starting index given by the function index_finder
# Iterate through every unlabeled rows
if start_index < len(new_labeled_df):
    print("\nIterating through remaining unlabelled rows...")

    non_labelled_rows = new_labeled_df.iloc[start_index:]

    progress_bar = tqdm(non_labelled_rows.iterrows(), total=len(non_labelled_rows), miniters=25)

    for index, row in progress_bar:
        try:
            prediction = labeller(row['parent_comment'], row['comment'])
            new_labeled_df.at[index, 'llama_temp0'] = prediction

            if index % 25 == 0:
                progress_bar.set_description(f"Processing Row {index}")

        except Exception as e:
            tqdm.write(f"Row {index} skipped due to error: {e}")
            continue

        if index % 10 == 0:
            new_labeled_df.to_csv(output_data_path, index=False)

    progress_bar.close()
    new_labeled_df.to_csv(output_data_path, index=False)

    print(f"Dataset labeling completed")
    print(f"Total rows processed: {len(non_labelled_rows)}")